# Day 15 – Structured Output – Extensive Solutions

Complete solutions for JSON mode, function calling, and error handling.

In [ ]:
import openai
import json
from dotenv import load_dotenv
import os

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

def ask_json(prompt, model="gpt-3.5-turbo"):
    """Force JSON output using response_format."""
    response = openai.ChatCompletion.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

print("Ready.")

## Exercise 1: Extract product information (name, price, stock status)

From messy product descriptions, extract structured data.

In [ ]:
descriptions = [
    "Apple iPhone 14, $999, in stock",
    "Samsung TV - 55 inches - $599.99 - out of stock",
    "Sony Headphones, price $199, available now"
]

def extract_product(desc):
    prompt = f"""Extract product name, price (as number), and stock status (in stock or out of stock) as JSON.
    Description: {desc}
    Output format: {{"name": string, "price": float, "stock": string}}"""
    try:
        return ask_json(prompt)
    except Exception as e:
        print(f"Error parsing: {e}")
        return None

for desc in descriptions:
    data = extract_product(desc)
    print(f"{desc} -> {data}")

## Exercise 2: SQL generation using function calling

Define a function `run_sql(query)` and ask the model to generate SQL for natural language requests.

In [ ]:
functions = [
    {
        "name": "run_sql",
        "description": "Execute a SQL query on the users table",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The SQL SELECT query to execute"
                }
            },
            "required": ["query"]
        }
    }
]

def generate_sql(natural_language):
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": natural_language}],
        functions=functions,
        function_call={"name": "run_sql"}
    )
    args = json.loads(response.choices[0].message.function_call.arguments)
    return args["query"]

queries = [
    "Show all users older than 18",
    "Get names and emails of users who signed up in 2023"
]
for q in queries:
    sql = generate_sql(q)
    print(f"Natural: {q}\nSQL: {sql}\n")

## Exercise 3: Parse conversation history into structured format

Given raw chat log, output JSON array with role and message.

In [ ]:
raw_chat = """
Alice: Hi, how are you?
Bob: I'm good, thanks! What about you?
Alice: Doing well. Let's discuss the project.
"""

def parse_conversation(raw_text):
    prompt = f"""Convert the following conversation into a JSON array of objects with 'role' and 'message'.
    Roles are the names before the colon.
    Conversation:
    {raw_text}
    Output only JSON."""
    return ask_json(prompt)

parsed = parse_conversation(raw_chat)
print(json.dumps(parsed, indent=2))

## Exercise 4: Error handling – retry on invalid JSON

Sometimes the model returns malformed JSON. We'll implement a retry loop.

In [ ]:
def robust_json_extract(prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = openai.ChatCompletion.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"}
            )
            return json.loads(response.choices[0].message.content)
        except json.JSONDecodeError as e:
            print(f"Attempt {attempt+1} failed: {e}. Retrying...")
            # Optionally modify prompt to be stricter
            prompt += "\nMake sure the output is valid JSON."
    raise Exception("Failed to extract valid JSON after retries")

# Test
data = robust_json_extract("Extract name and age from 'John is 30 years old'")
print(data)